# CNN Tuning with skorch + GridSearchCV

Wraps `LungCNN` in a scikit-learn-compatible `skorch.NeuralNetClassifier` so it can be tuned with the same `GridSearchCV`/`StratifiedKFold` pattern used for the MLP in `main.ipynb`. Loads the already-deduplicated clips from `HLS-CMDS/unique_lung_sounds.csv`/`HLS-CMDS/Unique/` instead of re-running the hashing step.

In [1]:
from pathlib import Path

import numpy as np, pandas as pd, librosa, torch
from torch import nn
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from skorch import NeuralNetClassifier
from skorch.callbacks import EarlyStopping
from skorch.dataset import ValidSplit

In [2]:
ROOT = Path("HLS-CMDS")
SAMPLE_RATE = 8000
HOP_LENGTH = 256
SECS = 10.0
N_MELS = 64
LENGTH_OF_FFT = 1024
RANDOM_STATE = 42

CLASSES = sorted(pd.read_csv(ROOT/"unique_lung_sounds.csv")["label"].unique())
LABEL2IDX = {c: i for i, c in enumerate(CLASSES)}

## Load data

Reads the deduplicated file list from `unique_lung_sounds.csv`, computes a log-mel spectrogram per clip, and stacks them into a single `(N, 1, N_MELS, time)` float32 array so it can be indexed/split like any other sklearn feature matrix.

In [3]:
def to_melspec(path):
    y, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    n = int(SECS*SAMPLE_RATE)
    y = np.pad(y, (0, n-len(y))) if len(y) < n else y[(len(y)-n)//2:(len(y)-n)//2+n]
    mel = librosa.feature.melspectrogram(y=y, sr=SAMPLE_RATE, n_fft=LENGTH_OF_FFT, hop_length=HOP_LENGTH, n_mels=N_MELS)
    m = librosa.power_to_db(mel, ref=np.max)
    return ((m - m.mean()) / (m.std() + 1e-6)).astype(np.float32)

unique_df = pd.read_csv(ROOT/"unique_lung_sounds.csv")

X = np.stack([
    to_melspec(ROOT/"Unique"/fname)[None, ...]
    for fname in unique_df["filename"]
]).astype(np.float32)
y = unique_df["label"].map(LABEL2IDX).to_numpy().astype(np.int64)

print("X shape:", X.shape)
print("classes:", CLASSES)

X shape: (94, 1, 64, 313)
classes: ['Coarse Crackles', 'Fine Crackles', 'Normal', 'Pleural Rub', 'Rhonchi', 'Wheezing']


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)

print(f"train samples: {len(X_train)}")
print(f"test samples:  {len(X_test)}")

train samples: 70
test samples:  24


## CNN

Same architecture as `main.ipynb`'s `LungCNN`: three conv blocks (`Conv2d -> BatchNorm2d -> ReLU`, maxpool on the first two, adaptive avg pool on the third), then a dropout + linear classifier head.

In [5]:
class LungCNN(nn.Module):
    def __init__(self, num_classes=len(CLASSES), dropout=0.30):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

## skorch wrapper + GridSearchCV

`NeuralNetClassifier` makes `LungCNN` behave like any other sklearn estimator (`fit`/`predict`/`get_params`), so `GridSearchCV` can tune it exactly like the MLP's `Pipeline` in `main.ipynb`. `module__*` params reach into `LungCNN.__init__`; the rest configure the skorch training loop (optimizer, lr, batch size, early stopping).

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"

net = NeuralNetClassifier(
    LungCNN,
    max_epochs=100,
    optimizer=torch.optim.Adam,
    criterion=nn.CrossEntropyLoss,
    callbacks=[("early_stop", EarlyStopping(patience=10))],
    device=device,
    verbose=0,
    train_split=ValidSplit(cv=0.2, stratified=True, random_state=RANDOM_STATE),
)

param_grid = {
    "lr": [1e-3, 1e-2],
    "optimizer__weight_decay": [1e-4, 1e-3],
    "module__dropout": [0.30, 0.50],
    "batch_size": [8, 16],
}

grid_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
grid = GridSearchCV(net, param_grid, cv=grid_cv, scoring="accuracy", n_jobs=1)
grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
print("Best CV accuracy: %.3f" % grid.best_score_)

Best params: {'batch_size': 8, 'lr': 0.001, 'module__dropout': 0.3, 'optimizer__weight_decay': 0.001}
Best CV accuracy: 0.686


## Evaluation

`grid.best_estimator_` is refit on the full training split; score it on the held-out test set.

In [7]:
best_cnn = grid.best_estimator_
cnn_pred = best_cnn.predict(X_test)

print(classification_report(y_test, cnn_pred, target_names=CLASSES, zero_division=0))
print(confusion_matrix(y_test, cnn_pred))

                 precision    recall  f1-score   support

Coarse Crackles       1.00      1.00      1.00         4
  Fine Crackles       1.00      0.67      0.80         3
         Normal       0.80      1.00      0.89         4
    Pleural Rub       1.00      1.00      1.00         5
        Rhonchi       0.67      0.50      0.57         4
       Wheezing       0.80      1.00      0.89         4

       accuracy                           0.88        24
      macro avg       0.88      0.86      0.86        24
   weighted avg       0.88      0.88      0.87        24

[[4 0 0 0 0 0]
 [0 2 0 0 1 0]
 [0 0 4 0 0 0]
 [0 0 0 5 0 0]
 [0 0 1 0 2 1]
 [0 0 0 0 0 4]]
